# Thursday: where does an invocation's time go?

*PhDOpen MIMUW, 15 October 2026. SeBS tutorial, part 1 of 2.*

**Take-away:** a function's latency is mostly not your function, and a local platform shows you mechanisms, not numbers.

Today you'll drive [SeBS](https://github.com/spcl/serverless-benchmarks) from Python: build a configuration, create a deployment, deploy benchmarks to **OpenWhisk** on your laptop and to your cloud account (**AWS Lambda** or **Google Cloud Functions**), invoke them, and take the measurements apart.

Everything runs from this notebook. `sebs_tutorial` (imported as `st`) only hides plumbing: checks, plots, and peeking into Docker, Kubernetes, and OpenWhisk. All SeBS operations are in the cells, so you can read, change, and reuse them.

## 0. Deploy OpenWhisk (During the mini-talk)

▶️ Run this cell as soon as the talk begins. It creates a Kubernetes cluster on your laptop (kind) and deploys OpenWhisk on it, from the images you copied into your local registry at home (`openwhisk/prepare.sh`): nothing is downloaded. It takes a few minutes; the cell waits until OpenWhisk is ready and reports progress. Listen to the talk meanwhile. If the cluster exists already, the cell only starts it. If you didn't run `prepare.sh`, the cell downloads the tools and images first (about 3 GB): it still works, just much more slowly.

In [ ]:
!./openwhisk/deploy.sh

## 1. Setup check

▶️ This checks Docker, SeBS, your OpenWhisk cluster, the local image registry, and the object storage. It also starts the object storage (RustFS) if needed. If OpenWhisk isn't ready yet, wait until the cell above finishes, then rerun this one.

In [ ]:
import time
import sebs_tutorial as st

st.check("thursday")

▶️ OpenWhisk itself runs as pods in your Kubernetes cluster: the controller (the API), the invoker (runs functions), Kafka between them, CouchDB for records, and a few helpers. `kubectl` lists them, with the kind node each one runs on. Rerun this cell during the next steps to see function pods appear.

In [ ]:
!kubectl --context kind-{st.KIND_CLUSTER} get pods -n openwhisk -o wide

*Optional:* to watch the pods live, open a JupyterLab terminal (File → New → Terminal), and in the tutorial directory run `export PATH=$PWD/openwhisk/deps:$PATH` and `kubectl get pods -n openwhisk -w`.

## 2. Probe the platform

Goal: follow one function from source code to a running container, and watch the platform react to invocations.

### The SeBS API in one picture

```
config (dict) ──► SeBS client ──► deployment client (OpenWhisk, GCP, AWS, ...)
                                       │
          benchmark (code + input) ────┤ get_function(): build, push, create
                                       ▼
                                   function ──► trigger.sync_invoke(input) ──► ExecutionResult
```

### Step 1: configuration, build, push, deploy

▶️ A SeBS configuration is plain JSON: an `experiments` part (language, runtime version, architecture) and a `deployment` part (platform settings). Load the course template and add the storage that `st.check` started.

In [ ]:
config = st.load_config("openwhisk")
config["deployment"]["openwhisk"]["storage"] = st.start_storage()
st.show_json(config)

Expand the JSON and find:
- `experiments.runtime`: Python 3.11; `experiments.system_variant`: `container`. On OpenWhisk, a function is a container image.
- `deployment.openwhisk.docker_registry.registry`: `localhost:5001`, a registry on your laptop. SeBS pushes images there and OpenWhisk pulls them from there.
- `deployment.openwhisk.storage.object.rustfs`: the object storage, [RustFS](https://rustfs.com) (an S3-compatible server) running in a Docker container. Note its two addresses (see below).

**Where is the storage?** The storage container runs on Docker's default bridge network. SeBS records its address there as `address` (`172.17.x.x:9000`); SeBS itself uses it, and so do functions on the same bridge (Friday's local backend). Your OpenWhisk functions run in pods inside kind, which sits on a different Docker network: they can't reach that address. So SeBS also records an `external_address`: your laptop's **external IP address** and the port that the storage publishes on the laptop (`9011`). OpenWhisk functions receive this one.

This is how you'd do it in practice. In a real deployment, the storage runs on another machine or VM, and the functions reach it at that machine's address and published port, ideally a stable one: a fixed IP or a DNS name. [SeBS's documentation](https://github.com/spcl/serverless-benchmarks/blob/master/docs/storage.md#network-configuration) describes the same setup.

**If your IP address changes** (other Wi-Fi, VPN, sleep and resume), deployed functions still point at the old address, and storage benchmarks fail with connection errors or timeouts. Rerun this cell and the next two, and call `get_function` again for each benchmark. SeBS compares the function's storage configuration with the new one and updates the action ("Updating function configuration due to changed storage configuration"). Without any network, there's no external IP; `st.start_storage` then falls back to an address that only works on your laptop.

▶️ Create the SeBS client and the OpenWhisk deployment client. `resource_prefix` names everything SeBS creates (storage buckets, functions).

In [ ]:
from sebs import SeBS
from sebs.faas.function import Trigger

sebs_client = SeBS(cache_dir="cache", output_dir="outputs", verbose=False, logging_filename=None)
openwhisk = sebs_client.get_deployment(config)
openwhisk.initialize(resource_prefix="phd")
experiment = sebs_client.get_experiment_config(config["experiments"])
print(openwhisk.name(), "resources:", openwhisk.config.resources.resources_id)

❓ **Before running the next cell:** what has to happen between "here is my Python code" and "the function answers"?

▶️ Now the benchmark: `110.dynamic-html` renders an HTML page from a template. `prepare_input` creates the function's input; `get_function` builds the image, pushes it, and creates the OpenWhisk action. Read the log lines: *Build … → Push … to registry: localhost:5001 → Creating new OpenWhisk action*.

In [ ]:
html = sebs_client.get_benchmark("110.dynamic-html", openwhisk, experiment)
html_input = html.prepare_input(openwhisk.system_resources, size="test")
fn_html = openwhisk.get_function(html)
print("function:", fn_html.name)
st.show_json(html_input)

✅ SeBS recorded which image it deployed. Ask OpenWhisk itself what it will run, and compare:

In [ ]:
info = st.action(fn_html)
print("SeBS:     ", fn_html.config.docker_image)
print("OpenWhisk:", info["image"], f"| memory {info['memory_mb']} MB | version {info['version']}")
if not info["image"].startswith(st.REGISTRY):
    print("✘ the action doesn't use the local registry: check docker_registry in the config")
elif info["image"] != fn_html.config.docker_image:
    print("✘ OpenWhisk runs another image than SeBS recorded: rerun get_function")
else:
    print("deployed ✔")

### Step 2: look into the registry

▶️ The registry speaks a simple HTTP API. List its repositories and image tags, then look at the image you just built: its size and its layers (newest first).

In [ ]:
print(st.registry_catalog())
st.registry_tags()

In [ ]:
image = info["image"]
print(f"image: {st.image_size(image)} MB, code package: {html.code_size / 1e3:.1f} KB")
st.image_layers(image)

❓ How large is the function image, and how large is your code within it?

### Step 3: invoke and read the platform's own record

▶️ A *trigger* invokes a function. We use an **HTTP trigger**, as a real client (or the cloud) would: SeBS makes the action a *web action*, reachable at a URL through OpenWhisk's API gateway (nginx), and sends it an HTTPS request with the input as JSON. (SeBS also has a *library* trigger, which calls the `wsk` command-line tool; its times include starting `wsk` for every invocation.)

In [ ]:
trigger = st.http_trigger(openwhisk, fn_html)
print(trigger.url)

▶️ Three invocations through SeBS. Every invocation returns an `ExecutionResult` with client-side times (including curl's: `http_connect`, `http_first_byte`), the function's output, and whether the container was cold.

In [ ]:
results = [trigger.sync_invoke(html_input) for _ in range(3)]
r = results[0]
print("client time:", r.times.client, "us | cold:", r.stats.cold_start, "| activation:", st.request_id(r))
# provider columns stay empty until we download the platform's metrics (section 3)
st.to_frame(results)

▶️ What did SeBS send? This is the same request, as a `curl` command. Run it and look at everything that comes back: the HTTP status and headers (OpenWhisk's own headers start with `x-openwhisk`), the body (the JSON that SeBS's wrapper in the function returns: the benchmark's `result`, the handler's `begin` and `end` timestamps, `is_cold`, and the `request_id`), and curl's timings.

In [ ]:
cmd = st.curl_command(trigger.url, html_input)
print(cmd, end="\n\n")
!{cmd}

❓ Which header carries the activation ID, and where does it appear in the body? How much of curl's total time was spent before the request was even sent (DNS, TCP, TLS)?

▶️ A pod now runs your function. Its Kubernetes events tell you whether the node had to pull the image, and how long it took.

In [ ]:
fn_pods = st.pods(fn_html)
display(fn_pods)
st.pod_events(fn_pods.name[0])

▶️ OpenWhisk keeps an *activation record* for every invocation (in CouchDB). The record includes `waitTime` (queueing in the platform), `initTime` (initializing a new container), and `duration` (running the action), all in ms.

In [ ]:
acts = st.activations(ids=[st.request_id(r) for r in results])
acts

✅ Which invocations initialized a container (have an `initTime`), and how many pods run your function now?

In [ ]:
has_init = acts.initTime.notna().tolist()
print(f"{sum(has_init)} of {len(has_init)} invocations initialized a container: {has_init}")
print(f"{len(st.pods(fn_html))} pod(s) run the function")
if not any(has_init):
    print("✘ none: the container was already warm. Force a cold start and rerun the last 3 cells:")
    print("   openwhisk.enforce_cold_start([fn_html], html)")

❓ How many of the three invocations were cold? Usually only the first one; sometimes the second one is, too, with a second pod. The requests were sent one after another: why could OpenWhisk start a second container? How long did pulling the image take, and which of `waitTime`, `initTime`, `duration` is missing on the warm invocations?

### Step 4: three kinds of start

| Start | How we get it | What it includes |
|---|---|---|
| Warm | invoke again | request only |
| Cold, image cached | `openwhisk.enforce_cold_start(...)` | container start + runtime init |
| Cold, image **not** cached | also remove the image from the nodes | image pull + container start + runtime init |

`enforce_cold_start` is part of SeBS's deployment API. On OpenWhisk, it changes an annotation of the action, which creates a new action version: the invoker never reuses containers of an old version. SeBS then deletes the old pods. `st.drop_image` runs `crictl rmi <image>` on every kind node.

▶️ Run the three cases:

In [ ]:
starts = {}
starts["warm"] = trigger.sync_invoke(html_input)

openwhisk.enforce_cold_start([fn_html], html)
starts["cold, image cached"] = trigger.sync_invoke(html_input)

openwhisk.enforce_cold_start([fn_html], html)
print("image removed from:", st.drop_image(fn_html))
starts["cold, image pulled"] = trigger.sync_invoke(html_input)

three = st.to_frame(starts.values())
three.insert(0, "kind", list(starts))
three = three.merge(st.activations(ids=three.request_id)[["activationId", "initTime"]],
                    left_on="request_id", right_on="activationId")
three[["kind", "client_time", "initTime", "internal_function_time"]]

✅

In [ ]:
t = three.client_time.tolist()
print("✔ warm < cold (cached) < cold (pull)" if t[0] < t[1] < t[2]
      else f"✘ client times {t}: rerun the cell (a single measurement is noisy!)")

❓ Clouds work hard to avoid the third case. How?

🚀 **Going further**
- Invoker logs: `!kubectl logs -n openwhisk owdev-invoker-0 --tail=30`. Find where the invoker starts a new container or reuses a warm one.
- Concurrency: `futures = [trigger.async_invoke(html_input) for _ in range(10)]`, then `st.pods(fn_html)`. How many pods appear?
- Edit the benchmark (`benchmarks/100.webapps/110.dynamic-html/python/function.py` in your SeBS directory, e.g. add a field to the result), and run `openwhisk.get_function(html)` again: SeBS notices the changed code, rebuilds, and pushes a new image. Find it with `st.registry_tags()`.
- Leave the function idle and rerun `st.pods(fn_html)` from time to time. How long does the pod stay warm?

## 3. Where does the time go, locally?

Four vantage points, each sees a different part of an invocation:

| Vantage point | Columns | Sees |
|---|---|---|
| Client | `client_time`, `http_connect`, `http_first_byte` | everything, from sending the request until the response arrives; curl's view: connection set up, first byte of the response |
| Provider metrics | `provider_wait`, `provider_init`, `provider_duration` | what the platform records: queueing, container init, action run (SeBS's `result.provider_times`: `waiting`, `initialization`, `execution`) |
| Function wrapper (SeBS) | `internal_function_time` | the handler, from its first to its last line |
| Benchmark internals | `download`, `compute`, `upload` | the phases the function measures itself |

All times are in milliseconds.

▶️ Deploy `210.thumbnailer`: it downloads an image from storage, resizes it, and uploads the thumbnail. Its function image was seeded into your registry, so there's nothing to build. `prepare_input` also uploads the input images to the storage.

In [ ]:
thumb = sebs_client.get_benchmark("210.thumbnailer", openwhisk, experiment)
thumb_input = thumb.prepare_input(openwhisk.system_resources, size="small")
fn_thumb = openwhisk.get_function(thumb)
thumb_trigger = st.http_trigger(openwhisk, fn_thumb)
st.to_frame([thumb_trigger.sync_invoke(thumb_input)])

✏️ **TODO 1: warm invocations.** Invoke the thumbnailer 30 times and collect the results. The next lines ask OpenWhisk for its records of these invocations (`download_metrics`, SeBS's API for provider metrics) and build a table.

In [ ]:
begin = time.time()
warm_results = []
# TODO: invoke thumb_trigger 30 times with thumb_input; append each result to warm_results

st.download_provider_metrics(openwhisk, fn_thumb, warm_results, begin, time.time())
warm = st.to_frame(warm_results)
warm.head()

✅

In [ ]:
cols = ["client_time", "provider_duration", "internal_function_time", "download", "compute", "upload"]
if len(warm) != 30:
    print(f"✘ {len(warm)} rows, expected 30")
elif warm.is_cold.iloc[1:].any():
    print("✘ cold starts after the first invocation: is someone else forcing cold starts?")
elif warm[cols].isna().any().any():
    print("✘ missing values in", list(warm[cols].columns[warm[cols].isna().any()]))
else:
    print("✔ 30 warm invocations, all four data points present")

✏️ **TODO 2: cold invocations.** Collect 5 cold invocations: force a cold start before each one.

*Hint:* `openwhisk.enforce_cold_start([fn_thumb], thumb)` (from Step 4).

In [ ]:
begin = time.time()
cold_results = []
# TODO: 5 times: force a cold start of fn_thumb, invoke, append the result to cold_results

st.download_provider_metrics(openwhisk, fn_thumb, cold_results, begin, time.time())
cold = st.to_frame(cold_results)
cold

✅

In [ ]:
if len(cold) == 5 and cold.is_cold.all() and (cold.provider_init > 0).all():
    print("✔ 5 cold starts, each with an initTime from OpenWhisk")
else:
    print(f"✘ {len(cold)} rows, cold: {cold.is_cold.tolist()}: did you call enforce_cold_start before each invocation?")

▶️ Medians per component, and the stacked breakdown:

In [ ]:
display(st.compare({"warm": warm, "cold": cold}))
st.plot_breakdown({"warm": warm, "cold": cold})

❓ Which components grow on a cold start? Why does the *wrapper* get slower, although the benchmark's own phases barely change?

❓ Download and upload are fast here. Why?

❓ Here you could subtract the function's timestamps from the client's. Why can't you in the cloud?

In [ ]:
st.save(warm, "results/thursday_openwhisk_warm.csv")
st.save(cold, "results/thursday_openwhisk_cold.csv")

## 4. The cloud: the same function, and a memory sweep

### Part 1: the same function in the cloud

Same API, different deployment: your AWS or GCP account (set up before the course, see README step 6; everything here fits into the free tier). Set `CLOUD` to `"aws"` or `"gcp"`; your credentials are in `configs/`. Nothing below depends on the cloud except the configuration and, in TODO 4, the price list. SeBS supports more platforms (e.g. Azure) with the same API and a different `configs/<platform>.json`.

▶️ Create the deployment client and deploy the thumbnailer. There's no OpenWhisk-style registry here: SeBS packages the code, and the cloud builds and runs it. The first run creates everything (storage buckets, the function): a few minutes. `st.cloud_prefix()` names your resources after your laptop, so students sharing an account don't collide.

In [ ]:
CLOUD = "aws"   # or "gcp": the cloud of your account

cloud_config = st.cloud_config(CLOUD)
cloud = sebs_client.get_deployment(cloud_config)
cloud.initialize(resource_prefix=st.cloud_prefix())
cloud_experiment = sebs_client.get_experiment_config(cloud_config["experiments"])

thumb_cloud = sebs_client.get_benchmark("210.thumbnailer", cloud, cloud_experiment)
thumb_cloud_input = thumb_cloud.prepare_input(cloud.system_resources, size="small")
fn_cloud = cloud.get_function(thumb_cloud)
cloud_trigger = st.http_trigger(cloud, fn_cloud)
print(fn_cloud.name, f"{fn_cloud.config.memory} MB", cloud_trigger.url)

▶️ 30 invocations, exactly as in TODO 1, over HTTPS this time. Friday reruns them.

In [ ]:
remote_begin = time.time()
remote_results = [cloud_trigger.sync_invoke(thumb_cloud_input) for _ in range(30)]
remote_end = time.time()
remote = st.to_frame(remote_results)
st.save(remote, f"results/thursday_{CLOUD}.csv")
st.compare({"local (OpenWhisk)": warm, f"remote ({CLOUD})": remote})

Provider columns are empty: clouds publish their logs with a delay (GCP: a few minutes, AWS: up to a minute). ▶️ Rerun this cell later (e.g. after the memory sweep):

In [ ]:
st.download_provider_metrics(cloud, fn_cloud, remote_results, remote_begin, remote_end)
remote = st.to_frame(remote_results)
st.save(remote, f"results/thursday_{CLOUD}.csv")
st.compare({"local (OpenWhisk)": warm, f"remote ({CLOUD})": remote})

✅

In [ ]:
ok = len(remote) == 30 and remote.client_time.median() > warm.client_time.median()
print("✔ 30 remote invocations, slower than local at the client" if ok else f"✘ {len(remote)} rows: rerun the invocation cell")

❓ Which components changed most between local and remote?

❓ Which of your local numbers would be misleading if reported as "serverless performance"?

### Part 2: memory sweep, three sizes

In both clouds, you choose the memory, and the CPU share grows with it:
- **GCP Cloud Functions (1st gen):** 256 MB gets 0.4 GHz, 1024 MB 1.4 GHz, 2048 MB 2.4 GHz;
- **AWS Lambda:** CPU proportional to memory, one full vCPU at 1,769 MB.

You'll run the thumbnailer at 256, 1024, and 2048 MB.

❓ **Predict before running:** with more memory, does the compute time drop? Does the cost?

✏️ **TODO 3: change the memory.** A function's settings live in `fn_cloud.config`. Complete `run_at`: set the memory, then push the new configuration to the cloud with the deployment client. Then run it for the three sizes (GCP: 1–2 minutes per size for the update, AWS: seconds).

*Hint:* `cloud.update_function_configuration(function, benchmark)`. The same call works for every platform.

In [ ]:
def run_at(memory_mb, n=20):
    # TODO: set the memory of fn_cloud and update its configuration in the cloud
    begin = time.time()
    results = [cloud_trigger.sync_invoke(thumb_cloud_input) for _ in range(n)]
    return results, begin, time.time()

MEMORY_SIZES = [256, 1024, 2048]
sweep_runs = {memory: run_at(memory) for memory in MEMORY_SIZES}

✅

In [ ]:
sweep = {memory: st.to_frame(results) for memory, (results, _, _) in sweep_runs.items()}
for memory, df in sweep.items():
    print(f"{memory:5d} MB: {len(df)} rows, median compute {df.compute.median():.1f} ms")
print("✔ memory updated" if fn_cloud.config.memory == MEMORY_SIZES[-1] else "✘ fn_cloud.config.memory doesn't change: check run_at")

▶️ Clouds bill the function's duration as they record it. Wait for the provider metrics (rerun the cell until it prints ✔):

In [ ]:
for memory, (results, begin, end) in sweep_runs.items():
    st.download_provider_metrics(cloud, fn_cloud, results, begin, end)
sweep = {memory: st.to_frame(results) for memory, (results, _, _) in sweep_runs.items()}
missing = sum(df.provider_duration.isna().sum() for df in sweep.values())
print("✔ provider metrics complete" if missing == 0 else f"✘ {missing} durations not published yet: wait a minute and rerun")

✏️ **TODO 4: the cost of one (warm) invocation** on your cloud, with $d$ the duration (`provider_duration`, ms) and $M$ the memory (MB):

| Cloud | Billed duration $t$ | Price |
|---|---|---|
| GCP (1st gen) | $d$ rounded **up to 100 ms** | $t \cdot \left(\frac{M}{1024} \cdot p_\text{GB-s} + \text{GHz} \cdot p_\text{GHz-s}\right)$, with `st.GCP_PRICE_PER_GB_S`, `st.GCP_PRICE_PER_GHZ_S`, and the CPU clock `st.GCP_CPU_GHZ[M]` |
| AWS Lambda | $d$ rounded **up to 1 ms** | $t \cdot \frac{M}{1024} \cdot p_\text{GB-s} + p_\text{request}$, with `st.AWS_PRICE_PER_GB_S` and `st.AWS_PRICE_PER_REQUEST` |

$t$ is in seconds. *Hint:* `math.ceil`.

In [ ]:
import math

def cost_per_invocation(duration_ms, memory_mb):
    return ...   # TODO: price of one invocation in USD on CLOUD

✅ ▶️ Check, then the sweep next to your laptop: the median compute time of OpenWhisk's warm runs (section 3) is the dashed line.

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    memory: {
        "compute_ms": df.compute.median(),
        "duration_ms": df.provider_duration.median(),
        "usd_per_million": cost_per_invocation(df.provider_duration.median(), memory) * 1e6,
    }
    for memory, df in sweep.items()
}).T
expected = [st.cloud_cost(CLOUD, d, m) * 1e6 for m, d in summary.duration_ms.items()]
ok = all(math.isclose(c, e, rel_tol=1e-6) for c, e in zip(summary.usd_per_million, expected))
print("✔ costs match the price list" if ok else f"✘ expected {[f'{e:.2f}' for e in expected]} USD per million: did you round the duration up?")
st.plot_memory_sweep(summary, local_compute_ms=warm.compute.median(), cloud=CLOUD)
summary

❓ Was your prediction right? Which memory size makes the cloud as fast as your laptop, and which is the cheapest per invocation?

❓ How much of your cost is rounding? Compare `duration_ms` with the billed duration.

🚀 **Going further:** add 128 and 512 MB to the sweep, or run it with `110.dynamic-html` (no storage).

## 5. Wrap-up

**For Friday:** keep your setup and the `results/` folder. The cloud function stays deployed. Here, we remove only the OpenWhisk functions; the cluster and the storage stay.

In [ ]:
st.cleanup_openwhisk(sebs_client, openwhisk)